# Random Forest Project

For this project we will be exploring publicly available data from [LendingClub.com](https://www.lendingclub.com/). Lending Club connects people who need money (borrowers) with people who have money (investors). We will try to create a model that will help predict whether a borrower paid back their loan in full.

We will use lending data from 2007-2010 and classify whether or not the borrower paid back their loan in full.

**Column descriptions:**
- `credit.policy`: 1 if customer meets LendingClub credit underwriting criteria
- `purpose`: Purpose of the loan
- `int.rate`: Interest rate of the loan (as a proportion)
- `installment`: Monthly installments owed
- `log.annual.inc`: Natural log of self-reported annual income
- `dti`: Debt-to-income ratio
- `fico`: FICO credit score
- `days.with.cr.line`: Days borrower has had a credit line
- `revol.bal`: Revolving balance
- `revol.util`: Revolving line utilization rate
- `inq.last.6mths`: Creditor inquiries in last 6 months
- `delinq.2yrs`: Times 30+ days past due in past 2 years
- `pub.rec`: Number of derogatory public records

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Get the Data

**Use pandas to read loan_data.csv as a dataframe called loans.**

In [ ]:
loans = pd.read_csv('loan_data.csv')
loans.head()

In [ ]:
loans.info()

In [ ]:
loans.describe()

# Exploratory Data Analysis

**Histogram of FICO distributions by credit.policy outcome.**

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
loans[loans['credit.policy']==1]['fico'].hist(bins=35, color='blue', label='Credit Policy=1', alpha=0.6, ax=ax)
loans[loans['credit.policy']==0]['fico'].hist(bins=35, color='red',  label='Credit Policy=0', alpha=0.6, ax=ax)
plt.xlabel('FICO')
plt.ylabel('Count')
plt.title('FICO Distribution by Credit Policy')
plt.legend()

**Similar histogram but selected by not.fully.paid column.**

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
loans[loans['not.fully.paid']==1]['fico'].hist(bins=35, color='blue', label='not.fully.paid=1', alpha=0.6, ax=ax)
loans[loans['not.fully.paid']==0]['fico'].hist(bins=35, color='red',  label='not.fully.paid=0', alpha=0.6, ax=ax)
plt.xlabel('FICO')
plt.ylabel('Count')
plt.title('FICO Distribution by Not Fully Paid')
plt.legend()

**Countplot showing counts of loans by purpose, with hue defined by not.fully.paid.**

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(x='purpose', hue='not.fully.paid', data=loans, palette='RdBu')
plt.title('Loan Count by Purpose')
plt.xticks(rotation=45)

**Jointplot of FICO score vs interest rate.**

In [ ]:
sns.jointplot(x='fico', y='int.rate', data=loans, color='purple', alpha=0.3)

**lmplot to see the trend between FICO and interest rate, separated by not.fully.paid and credit.policy.**

In [ ]:
sns.lmplot(x='fico', y='int.rate', data=loans, hue='credit.policy',
           col='not.fully.paid', palette='Set1',
           scatter_kws={'alpha':0.3, 's':10}, line_kws={'lw':2})

# Setting up the Data

## Categorical Features

The `purpose` column is categorical. We need to transform it using dummy variables.

**Create cat_feats list and use pd.get_dummies to create final_data.**

In [ ]:
loans.info()

In [ ]:
cat_feats = ['purpose']
final_data = pd.get_dummies(loans, columns=cat_feats, drop_first=True)
final_data.head()

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = final_data.drop('not.fully.paid', axis=1)
y = final_data['not.fully.paid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

## Training a Decision Tree Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)

## Predictions and Evaluation of Decision Tree

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

dt_pred = dtree.predict(X_test)

print('Confusion Matrix:')
print(confusion_matrix(y_test, dt_pred))
print('\nClassification Report:')
print(classification_report(y_test, dt_pred))

## Training the Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=200)
rfc.fit(X_train, y_train)

## Predictions and Evaluation of Random Forest

In [ ]:
rf_pred = rfc.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, rf_pred))

In [ ]:
print('Confusion Matrix:')
print(confusion_matrix(y_test, rf_pred))

## Feature Importances

In [ ]:
feat_imp = pd.Series(rfc.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(12,6))
feat_imp.plot(kind='bar', color='steelblue')
plt.title('Random Forest Feature Importances')
plt.ylabel('Importance')
plt.tight_layout()

## What performed better — Random Forest or Decision Tree?

**Random Forest** achieved higher overall accuracy (**85%** vs **73%**), but the comparison is nuanced:

| Metric | Decision Tree | Random Forest |
|---|---|---|
| Accuracy | 73% | 85% |
| Class 0 F1 | 0.84 | 0.92 |
| Class 1 F1 | 0.22 | 0.06 |
| Class 1 Recall | 0.24 | 0.03 |

The Random Forest is much better at predicting loans that **will** be fully paid back (class 0), but it almost completely misses defaulters (class 1 recall = 0.03). This is because the dataset is imbalanced (~84% fully paid), so the forest learns to predict the majority class.

**For an investor, identifying risky loans (class 1) matters most.** Techniques like `class_weight='balanced'`, SMOTE oversampling, or tuning the probability threshold would improve recall for class 1 in a real-world scenario.
